# big CM MD->QM project results compilation and plotting

**Please avoid working in the original version in the DeYonker-lab-scripts repo! Copy to your own QM dir and use your local version**

Note from DAW 2026-09-30: cleaned this up for Nate to use to get latest data for his talks! cut down to just the plots of all three active site sets together, removed the batch model stuff bc it's not relevant, and added stats/the plots Nate wants to have. also increased the font size so it'll be more visible in a talk.

For more ideas of how you can work with the full data set for the paper/presentations/etc, also look at CM_MD-to-QM_paper.ipynb

--------------------

This notebook pulls together everyone's data for the three active sites so it can all be plotted together.

|             | ligand | active site   | frames      | model type | parent directory for project                        |
|-------------|--------|---------------|-------------|------------|-----------------------------------------------------|
| **Dr D.**   | A:128  | A/C interface | all         | individual | /project/ndyonker/chem/cm-MD-processing/MD-QM/QM-A  |
| **Pedro**   | B:256  | B/A interface | all         | individual | /project/pssntngr/chem/chorismate_mutase/QM-B/      |
| **Haritha** | C:384  | C/B interface | all         | individual | /project/hksasi/chem/chorismate_mutase/QM-C/        |

# Notebook environment setup: packages and default font sizes


In [34]:
import os, os.path, glob
import pickle
import ast
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from matplotlib.lines import Line2D
import seaborn as sns
import tol_colors as tc

The following block sets up some default figure widths and font sizes to make it easier to make nice consistent figures for publication. We don't have to use them from day one but I figured it's nice to already have in here for later.

fig sizes for publications:
- single column: approx 8cm wide
- double column: approx 18cm wide


In [35]:
# set default font sizes
# use 16/18/20 for publication quality figures with widths of 8 or 18
SMALL_SIZE = 16 
MEDIUM_SIZE = 18 
BIGGER_SIZE = 20 

# going to be making smaller plots for initial looks otherwise the figs are huge on a big screen. so halving the font sizes to match
#SMALL_SIZE = 12
#MEDIUM_SIZE = 12 
#BIGGER_SIZE = 16 

plt.rc('font', size=SMALL_SIZE)          # controls default text sizes
plt.rc('axes', titlesize=MEDIUM_SIZE)     # fontsize of the axes title
plt.rc('axes', labelsize=MEDIUM_SIZE)    # fontsize of the x and y labels
plt.rc('xtick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
plt.rc('ytick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
plt.rc('legend', fontsize=SMALL_SIZE)    # legend fontsize
plt.rc('figure', titlesize=BIGGER_SIZE)  # fontsize of the figure title

# single column figure width
scolw = 8
# double column figure width
dcolw = 18

# Read in data

In [36]:
data = {}

# get data for active site A 
pathA = '/project/ndyonker/chem/cm-MD-processing/MD-QM/QM-A'
QMAfiles = glob.glob(f'{pathA}/*/*_results.csv')
if QMAfiles:
    QMAfiles.sort()
    for f in QMAfiles:
        if QMAfiles.index(f)==0:
            data['A_ind'] = pd.read_csv(f,index_col='frame')
        else:
            data['A_ind'] = pd.concat([data['A_ind'],pd.read_csv(f,index_col='frame')])

# get data for active site B 
pathB = '/project/pssntngr/chem/chorismate_mutase/QM-B/'
QMBfiles = glob.glob(f'{pathB}/*/*_results.csv')
if QMBfiles:
    QMBfiles.sort()
    for f in QMBfiles:
        if QMBfiles.index(f)==0:
            data['B_ind'] = pd.read_csv(f,index_col='frame')
        else:
            data['B_ind'] = pd.concat([data['B_ind'],pd.read_csv(f,index_col='frame')])

# get data for active site C
pathC = '/project/hksasi/chem/chorismate_mutase/QM-C/'
QMCfiles = glob.glob(f'{pathC}/*/*_results.csv')
if QMCfiles:
    QMCfiles.sort()
    for f in QMCfiles:
        if QMCfiles.index(f)==0:
            data['C_ind'] = pd.read_csv(f,index_col='frame')
        else:
            data['C_ind'] = pd.concat([data['C_ind'],pd.read_csv(f,index_col='frame')])

# combine all individual models into one big data set and call this df
data['all_ind'] = pd.concat([data['A_ind'],data['B_ind'],data['C_ind']])
df = data['all_ind']


In [ ]:
# show which data sets we've got some values for
data.keys()

In [38]:
# define labels, max and min values so that we can easily set axes titles/legend values and axes limits consistently in plots
# keys of this dict are the column labels from the csv file/dataframes

plotopts = {'dGa': {'longlabel': r'$\Delta G ^\ddagger$ (kcal/mol)',
                'shortlabel': r'$\Delta G ^\ddagger$',
                'max': max([data[key]['dGa'].max() for key in data.keys()]), 
                'min': min([data[key]['dGa'].min() for key in data.keys()])
                },
        'dGr': {'longlabel': r'$\Delta G _{rxn}$ (kcal/mol)',
                'shortlabel': r'$\Delta G _{rxn}$',
                'max': max([data[key]['dGr'].max() for key in data.keys()]), 
                'min': min([data[key]['dGr'].min() for key in data.keys()])
                },
        'size': {'longlabel': 'model size (no. atoms)',
                'shortlabel': 'size',
                'max': max([data[key]['size'].max() for key in data.keys()]), 
                'min': min([data[key]['size'].min() for key in data.keys()])
                },
        'charge': {'longlabel': 'overall charge',
                'shortlabel': 'charge',
                'max': max([data[key]['charge'].max() for key in data.keys()]), 
                'min': min([data[key]['charge'].min() for key in data.keys()])
                },
        'fnum': {'longlabel': 'frame used to create model',
                'shortlabel': 'frame',
                'max': max([data[key]['fnum'].max() for key in data.keys()]), 
                'min': min([data[key]['fnum'].min() for key in data.keys()])
                }
        }

ligands = ['A:128','B:256','C:384']

# Original plots of all active sites together

## as separate subplots

In [ ]:
colours = [tc.high_contrast.red,tc.high_contrast.yellow,tc.high_contrast.blue]
markers = ['o','^','s']
ligands=['A:128','B:256','C:384']

legicons = []
fig, axs = plt.subplots(nrows=3,ncols=4,figsize=(12,7),layout='constrained')
for m, mod in enumerate(['A_ind','B_ind','C_ind']):
    if mod in data.keys():
        for i, ydata in enumerate(['dGa','dGr','size','charge']):
            axs[m,i].axhline(y=0,color='black',linewidth=1)
            data[mod].plot(x='fnum',y=ydata,kind='scatter',ax=axs[m,i],color=colours[m],marker=markers[m])
            axs[m,i].set_ylim(bottom=plotopts[ydata]['min']-1,top=plotopts[ydata]['max']+1)
            axs[m,i].set_xlabel(None)
            axs[m,i].set_ylabel(plotopts[ydata]['longlabel'])
        legicons.append(matplotlib.lines.Line2D([], [], linewidth=0, color=colours[m], marker=markers[m], markersize=10, label=ligands[m]))

fig.supxlabel(plotopts['fnum']['longlabel'])
#fig.suptitle('all active sites,individual models\n')
fig.legend(handles=legicons, loc='outside upper center', ncols=3)
fig.align_labels()

In [ ]:
colours = [tc.high_contrast.red,tc.high_contrast.yellow,tc.high_contrast.blue]
markers = ['o','^','s']
ligands=['A:128','B:256','C:384']

legicons = []
fig, axs = plt.subplots(nrows=3,ncols=4,figsize=(12,7),layout='constrained')
for m, mod in enumerate(['A_ind','B_ind','C_ind']):
    if mod in data.keys():
        for i, ydata in enumerate(['dGa','dGr']):
            axs[m,i].axhline(y=0,color='black',linewidth=1)
            data[mod].plot(x='size',y=ydata,kind='scatter',ax=axs[m,i],color=colours[m],marker=markers[m])
            axs[m,i].set_ylim(bottom=plotopts[ydata]['min']-1,top=plotopts[ydata]['max']+1)
            axs[m,i].set_xlim(left=plotopts['size']['min']-1,right=plotopts['size']['max']+1)
            axs[m,i].set_ylabel(plotopts[ydata]['longlabel'])
            axs[m,i+2].axhline(y=0,color='black',linewidth=1)
            data[mod].plot(x='charge',y=ydata,kind='scatter',ax=axs[m,i+2],color=colours[m],marker=markers[m])
            axs[m,i+2].set_ylim(bottom=plotopts[ydata]['min']-1,top=plotopts[ydata]['max']+1)
            axs[m,i+2].set_xlim(left=plotopts['charge']['min']-1,right=plotopts['charge']['max']+1)
            axs[m,i+2].set_ylabel(plotopts[ydata]['longlabel'])
            if m==2:
                axs[m,i].set_xlabel(plotopts['size']['longlabel'])
                axs[m,i+2].set_xlabel(plotopts['charge']['longlabel'])
            else:
                axs[m,i].set_xlabel(None)
                axs[m,i+2].set_xlabel(None)
        legicons.append(matplotlib.lines.Line2D([], [], linewidth=0, color=colours[m], marker=markers[m], markersize=10, label=ligands[m]))

#fig.suptitle('all active sites,individual models\n')
fig.legend(handles=legicons, loc='outside upper center', ncols=3)
fig.align_labels()

In [ ]:
ligands = ['A:128','B:256','C:384']
cols = [tc.high_contrast.red,tc.high_contrast.yellow,tc.high_contrast.blue]
fig, axs = plt.subplots(nrows=3,ncols=4,figsize=(12,7),layout='constrained')
for i, mod in enumerate(['A_ind','B_ind','C_ind']):
    if mod in data.keys():
        for j, yv in enumerate(['dGa','dGr','size','charge']):
            if yv in ['size','charge']:
                binrange = np.arange(np.floor(plotopts[yv]['min'])-0.5, np.ceil(plotopts[yv]['max'])+1.5,1)
            else:
                binrange = np.arange(np.floor(plotopts[yv]['min']), np.ceil(plotopts[yv]['max'])+1,1)
            data[mod].hist(column=yv,
                        ax=axs[i,j],
                        bins=binrange,
                        legend=False, 
                        grid=False,
                        color=cols[i])
            axs[i,j].set_title(None)
            if j == 0:
                axs[i,j].set_ylabel(ligands[i])
            else:
                axs[i,j].set_ylabel(None)
            if i == 2:
                axs[i,j].set_xlabel(plotopts[yv]['longlabel'])
            else:
                axs[i,j].set_xlabel(None)

fig.align_labels()

## overlaid

In [ ]:
# results by frame, plotted with different colours for each active site

colours = [tc.high_contrast.red,tc.high_contrast.yellow,tc.high_contrast.blue]
markers = ['o','^','s']
ligands=['A:128','B:256','C:384']

legicons = []
fig, axs = plt.subplots(nrows=1,ncols=4,figsize=(12,3.5),layout='constrained')
for m, mod in enumerate(['A_ind','B_ind','C_ind']):
    if mod in data.keys():
        for i, ydata in enumerate(['dGa','dGr','size','charge']):
            axs[i].axhline(y=0,color='black',linewidth=1)
            data[mod].plot(x='fnum',y=ydata,kind='scatter',ax=axs[i],color=colours[m],marker=markers[m])
            axs[i].set_ylim(bottom=plotopts[ydata]['min']-1,top=plotopts[ydata]['max']+1)
            axs[i].set_xlabel(None)
            axs[i].set_ylabel(plotopts[ydata]['longlabel'])
        legicons.append(matplotlib.lines.Line2D([], [], linewidth=0, color=colours[m], marker=markers[m], markersize=10, label=ligands[m]))

fig.supxlabel(plotopts['fnum']['longlabel'])
#fig.suptitle('all active sites,individual models\n')
fig.legend(handles=legicons, loc='outside upper center', ncols=3)
fig.align_labels()

In [ ]:
colours = [tc.high_contrast.red,tc.high_contrast.yellow,tc.high_contrast.blue]
markers = ['o','^','s']
ligands=['A:128','B:256','C:384']
legicons=[]

fig, axs = plt.subplots(nrows=3,ncols=3,figsize=(8,7),layout='constrained')
axs[0,2].axis("off")

for m, mod in enumerate(['A_ind','B_ind','C_ind']):
    if mod in data.keys():
        for i, ydata in enumerate(['dGa','dGr']):
            for j, xdata in enumerate(['size','charge']):
                axs[i+1,j].axhline(y=0,color='black',linewidth=1)
                data[mod].plot(x=xdata,y=ydata,kind='scatter',ax=axs[i+1,j],color=colours[m],marker=markers[m])
                axs[i+1,j].set_ylim(bottom=plotopts[ydata]['min']-1,top=plotopts[ydata]['max']+1)
                axs[i+1,j].set_xlim(left=plotopts[xdata]['min']-1,right=plotopts[xdata]['max']+1)
                axs[i+1,j].set_xlabel(None)
                axs[i+1,j].set_ylabel(None)
                # histograms of size/charge
                if i==0:
                    binrange = np.arange(np.floor(plotopts[xdata]['min'])-0.5, np.ceil(plotopts[xdata]['max'])+1.5,1)
                    data[mod].hist(column=xdata,
                                ax=axs[i,j],
                                bins=binrange,
                                legend=False, 
                                grid=False,
                                color=colours[m],
                                alpha=0.5)
                    #axs[i,j].set_ylim(bottom=0,top=100)
                    axs[i,j].set_title(None)
            # histograms of deltaGs
            binrange = np.arange(np.floor(plotopts[ydata]['min']), np.ceil(plotopts[ydata]['max'])+1,1)
            data[mod].hist(column=ydata,
                                ax=axs[i+1,2],
                                bins=binrange,
                                legend=False, 
                                grid=False,
                                color=colours[m],
                                orientation='horizontal',
                                alpha=0.5)
            #axs[i+1,2].set_xlim(left=0,right=100)
            axs[i+1,2].set_title(None)
        legicons.append(matplotlib.lines.Line2D([], [], linewidth=0, color=colours[m], marker=markers[m], markersize=10, label=ligands[m]))

axs[1,0].set_ylabel(plotopts['dGa']['longlabel'])
axs[2,0].set_ylabel(plotopts['dGr']['longlabel'])
axs[2,0].set_xlabel(plotopts['size']['longlabel'])
axs[2,1].set_xlabel(plotopts['charge']['longlabel'])

axs[0,0].set_ylabel('number of models')
axs[2,2].set_xlabel('number of models')

axs[0,2].legend(handles=legicons, loc='center')
fig.align_labels()

## combined into one set

In [ ]:
colours = [tc.medium_contrast.dark_blue,tc.medium_contrast.dark_red,tc.bright.green,tc.bright.yellow]
markers = ['o','^','s','d']

fig, axs = plt.subplots(nrows=1,ncols=4,figsize=(12,3),layout='constrained')
for i, ydata in enumerate(['dGa','dGr','size','charge']):
    axs[i].axhline(y=0,color='black',linewidth=1)
    data['all_ind'].plot(x='fnum',y=ydata,kind='scatter',ax=axs[i],color=colours[i],marker=markers[i])
    axs[i].set_ylim(bottom=plotopts[ydata]['min']-1,top=plotopts[ydata]['max']+1)
    axs[i].set_xlabel(None)
    axs[i].set_ylabel(plotopts[ydata]['longlabel'])

fig.supxlabel(plotopts['fnum']['longlabel'])
fig.suptitle('all active sites, individual models')
fig.align_labels()

In [ ]:
colours = [tc.medium_contrast.dark_blue,tc.medium_contrast.dark_red,tc.bright.green,tc.bright.yellow]
markers = ['o','^']

fig, axs = plt.subplots(nrows=3,ncols=3,figsize=(8,7),layout='constrained')
axs[0,2].axis("off")

for i, ydata in enumerate(['dGa','dGr']):
    for j, xdata in enumerate(['size','charge']):
        axs[i+1,j].axhline(y=0,color='black',linewidth=1)
        data['all_ind'].plot(x=xdata,y=ydata,kind='scatter',ax=axs[i+1,j],color=colours[i],marker=markers[i])
        axs[i+1,j].set_ylim(bottom=plotopts[ydata]['min']-1,top=plotopts[ydata]['max']+1)
        axs[i+1,j].set_xlim(left=plotopts[xdata]['min']-1,right=plotopts[xdata]['max']+1)
        axs[i+1,j].set_xlabel(None)
        axs[i+1,j].set_ylabel(None)
        # histograms of size/charge
        if i==0:
            binrange = np.arange(np.floor(plotopts[xdata]['min'])-0.5, np.ceil(plotopts[xdata]['max'])+1.5,1)
            data['all_ind'].hist(column=xdata,
                        ax=axs[i,j],
                        bins=binrange,
                        legend=False, 
                        grid=False,
                        color=colours[j+2])
            #axs[i,j].set_ylim(bottom=0,top=100)
            axs[i,j].set_title(None)
    # histograms of deltaGs
    binrange = np.arange(np.floor(plotopts[ydata]['min']), np.ceil(plotopts[ydata]['max'])+1,1)
    data['all_ind'].hist(column=ydata,
                        ax=axs[i+1,2],
                        bins=binrange,
                        legend=False, 
                        grid=False,
                        color=colours[i],
                        orientation='horizontal')
    #axs[i+1,2].set_xlim(left=0,right=100)
    axs[i+1,2].set_title(None)

axs[1,0].set_ylabel(plotopts['dGa']['longlabel'])
axs[2,0].set_ylabel(plotopts['dGr']['longlabel'])
axs[2,0].set_xlabel(plotopts['size']['longlabel'])
axs[2,1].set_xlabel(plotopts['charge']['longlabel'])

axs[0,0].set_ylabel('number of models')
axs[2,2].set_xlabel('number of models')

fig.suptitle('all active sites, individual models')
fig.align_labels()

# New plots 2026-09-30

## print stats for full set and each active site

In [ ]:
# define std as std with ddof=0 so it's not using ddof=1 unintentionally
def std(x):
    return x.std(ddof=0)

# get stats across models that are done, round to 1 dp, T to transpose/flip orientation
df.loc[df['done']=='Y',['dGa','dGr']].apply(['count','median','mean',std,'min','max','ptp']).round(1).T

In [ ]:
# get stats across models that are done, round to 1 dp, T to transpose/flip orientation
print('stats for QM-A models')
print(data['A_ind'].loc[data['A_ind']['done']=='Y',['dGa','dGr']].apply(['count','median','mean',std,'min','max','ptp']).round(1).T)

print('\nstats for QM-B models')
print(data['B_ind'].loc[data['B_ind']['done']=='Y',['dGa','dGr']].apply(['count','median','mean',std,'min','max','ptp']).round(1).T)

print('\nstats for QM-C models')
print(data['C_ind'].loc[data['C_ind']['done']=='Y',['dGa','dGr']].apply(['count','median','mean',std,'min','max','ptp']).round(1).T)

In [47]:
### save vals as variables for printing later ###
dGa_mean_std = f'{df["dGa"].mean().round(1)} ± {df["dGa"].std(ddof=0).round(1)}'
dGr_mean_std = f'{df["dGr"].mean().round(1)} ± {df["dGr"].std(ddof=0).round(1)}'

## activation energy vs reaction energy

In [ ]:
fig, ax = plt.subplots(figsize=(12,8),layout='constrained')
ax.axhline(y=0,color='black',linewidth=1)
# plot all points in grey
df.plot(ax=ax, kind='scatter',x='dGa',y='dGr', color=[0.8,0.8,0.8])
# means
ax.axhline(y=df['dGr'].mean(),color='black',linewidth=1, linestyle='--',zorder=0)
ax.axvline(x=df['dGa'].mean(),color='black',linewidth=1, linestyle='--',zorder=0)

# isolate ones where any wat rmsd is > 1
plotdf = df.loc[(df['rms_ts-p_wat']>1)|(df['rms_ts-r_wat']>1)|(df['rms_p-r_wat']>1),:]
# r more like ts than p is - product overstabilised? - colour blue
plotdf.loc[plotdf['rms_ts-r_all']<plotdf['rms_ts-p_all'],:].plot(ax=ax, kind='scatter',x='dGa',y='dGr',s=30,color=tc.high_contrast.blue)
# p more like ts than r is - reactant overstabilised? - colour red
plotdf.loc[plotdf['rms_ts-r_all']>plotdf['rms_ts-p_all'],:].plot(ax=ax, kind='scatter',x='dGa',y='dGr',s=30,color=tc.high_contrast.red)
# r and p more like each other than ts - both overstabilised? - colour yellow
### COMMENT LINE BELOW OUT TO JUST HAVE RED AND BLUE DOTS LIKE I RECOMMENDED FOR PREVIOUS TALKS?? ###
plotdf.loc[(plotdf['rms_ts-r_all']>plotdf['rms_p-r_all'])&(plotdf['rms_ts-p_all']>plotdf['rms_p-r_all']),:].plot(ax=ax, kind='scatter',x='dGa',y='dGr',s=30,color=tc.high_contrast.yellow)

ax.set_ylabel(plotopts['dGr']['longlabel'])
ax.set_xlabel(plotopts['dGa']['longlabel'])
ax.grid(axis='both',which='major',linestyle=':')
ax.set_axisbelow(True)
ax.set_title(f'mean {plotopts["dGa"]["shortlabel"]} = {dGa_mean_std} kcal/mol\nmean {plotopts["dGr"]["shortlabel"]} = {dGr_mean_std} kcal/mol',size=MEDIUM_SIZE)

## overlaid violins of dGa

In [ ]:
import matplotlib.patches as mpatches

##### plot #####
cols = [tc.high_contrast.red,tc.high_contrast.yellow,tc.high_contrast.blue]
fig, ax = plt.subplots(figsize=(8,8),layout='constrained')

dfs = []
for i, key in enumerate(['A_ind','B_ind','C_ind']):
    #note that matplotlib violins hate nans so you need to filter out rows without energy vals
    selrows = data[key]['done'] == 'Y'
    plotdf = data[key].loc[selrows,:]
    dfs.append(plotdf)
    # first "layer" = semi-transparent fill
    parts1 = ax.violinplot(plotdf['dGa'],positions=[1],side='low', showextrema=False, showmeans=False) #widths=0.2, 
    for pc in parts1['bodies']:
        pc.set_alpha(0.3)
        pc.set_edgecolor(cols[i])
        pc.set_facecolor(cols[i])
        pc.set_zorder(4-i) #layer in reverse order - blue is last but darkest so harder to see everything when it's on top
    # second layer: opaque outlines, no fill
    parts2 = ax.violinplot(plotdf['dGa'],positions=[1],side='low', showextrema=False, showmeans=True) #widths=0.2, 
    for pc in parts2['bodies']:
        pc.set_alpha(1)
        pc.set_linewidth(2)
        pc.set_edgecolor(cols[i])
        pc.set_facecolor("None")
        pc.set_zorder(4-i)
    for partname in ['cmeans']:
        vp = parts2[partname]
        vp.set_edgecolor(cols[i])
        vp.set_facecolor(cols[i])
        vp.set_linewidth(2)
        vp.set_zorder(4-i)

#allcol = 'black'
allcol = '#008816'
plotdf = pd.concat(dfs)
parts1 = ax.violinplot(plotdf['dGa'],positions=[1],side='high', showextrema=False, showmeans=False) #widths=0.6, 
for pc in parts1['bodies']:
    pc.set_alpha(0.3)
    pc.set_edgecolor(allcol)
    pc.set_facecolor(allcol)
    pc.set_zorder(5)    # topmost layer
parts2 = ax.violinplot(plotdf['dGa'],positions=[1],side='high', showextrema=False, showmeans=True) #widths=0.6, 
for pc in parts2['bodies']:
    pc.set_alpha(1)
    pc.set_edgecolor(allcol)
    pc.set_facecolor("none")
    pc.set_linewidth(2)
    pc.set_zorder(5)
for partname in ['cmeans']:
    vp = parts2[partname]
    vp.set_edgecolor(allcol)
    vp.set_facecolor(allcol)
    vp.set_linewidth(2)
    vp.set_zorder(5)

# create example opaque outline/translucent fill rectangles and use as legend icons
activesites = ['A/C*','B/A*','C/B*']
patches=[0,0,0,0]
for i, c in enumerate(cols):
    patches[i] = mpatches.Patch(facecolor=(c,0.2), edgecolor=(c,1), linewidth=2, label=activesites[i])
patches[3] = mpatches.Patch(facecolor=(allcol,0.2), edgecolor=(allcol,1), linewidth=2, label='all')
plt.legend(handles=patches, loc='upper right', title='active site', fontsize=20, title_fontsize=20)

ax.axhline(y=15.4, color='black', linestyle='--', zorder=1, linewidth=2)
ax.annotate(r'expt. $\Delta G ^\ddagger$',xy=(1.48,15.6),ha='right',va='bottom', fontsize=20)

ax.set_xlim(left=0.7,right=1.5)
ax.set_ylim(bottom=plotopts['dGa']['min']-1,top=plotopts['dGa']['max']+1)
ax.set_xlabel(None)
ax.set_ylabel(plotopts['dGa']['longlabel'], fontsize=20)
ax.grid(axis='y', which='major', linestyle=':')
ax.set_axisbelow(True)
ax.set_xticks([])
ax.set_xticklabels([])

fig.align_labels()